# 04 — Synthesis and benchmark coverage

Audit planned conditions against the saved synthesis records and native benchmark provenance. The database is opened read-only. Change `DATABASE_PATH` and `RESULTS_PATH` together when inspecting a separate study.

In [11]:
import os
import sqlite3
import sys
from functools import partial
from itertools import product
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sqlalchemy import and_, create_engine, or_, select, tuple_
from sqlalchemy.orm import sessionmaker

cwd = Path.cwd()
root = next((p for p in (cwd, *cwd.parents) if (p / "src/shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the project.")
for import_path in (root, root / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from benchmark.application.select_champions import ChampionSelectionService
from benchmark.application.trials.audit import BenchmarkAuditService, SynthesisConditionSpec
from benchmark.application.trials.workload import BenchmarkWorkload
from benchmark.infra.io.code_reader import FilesystemCodeReader
from benchmark.infra.io.trace_repository import BenchmarkStateRepository
from benchmark.infra.logging import BenchmarkLogger
from benchmark.infra.storage import ChampionsReadRepository, SQLiteSynthesisReadRepository
from evolution.application.synthesis.generation_analysis import analyze_generations
from evolution.infra.storage.synthesis.repository import SQLiteSynthesisRepository
from notebooks.analysis.plotting.style import FONT_FAMILY, get_solver_color
from shared.infra.database.tables import ExperimentORM

DATABASE_PATH = Path(os.environ.get("AAD_SYNTHESIS_DB", root / "data/db.sqlite3"))
RESULTS_PATH = root / "results"
GAP_MODEL = None  # Optional exact model or classical solver ID for the gap tables.

In [12]:
problems = (1, 8, 11, 15, 21)
modes = ("explicit", "implicit")
qwen_models = (
    "qwen2.5-coder-14b-instruct-q4_k_m.gguf",
    "qwen2.5-coder-32b-instruct-q4_k_m.gguf",
    "gwdg.qwen3-coder-next",
)
luna_model = "openai.gpt-6-luna"
qwen_plan = tuple(
    SynthesisConditionSpec(*values)
    for values in product(
        problems, (2, 10, 50), (0.0, 0.05, 0.1, 0.2), modes,
        ("baseline", "thinking", "vectorization", "guided"),
    )
)
luna_plan = tuple(
    SynthesisConditionSpec(*values)
    for values in product(problems, (10, 50), (0.0, 0.2), modes, ("baseline", "vectorization"))
)
planned_models = dict.fromkeys(qwen_models, qwen_plan) | {luna_model: luna_plan}
baseline_ids = ("cmaes", "de")
baseline_targets = tuple(sorted({(c.dim, c.noise_std, c.problem_id) for c in qwen_plan}))

database_path = (root / DATABASE_PATH).resolve()
uri = f"{database_path.as_uri()}?mode=ro"
engine = create_engine("sqlite://", creator=partial(sqlite3.connect, uri, uri=True))
try:
    open_session = sessionmaker(engine)
    synthesis_reader = SQLiteSynthesisReadRepository(open_session)
    workload = BenchmarkWorkload(
        sqlite_repo=synthesis_reader,
        champion_selection=ChampionSelectionService(ChampionsReadRepository(open_session)),
        state_repo=BenchmarkStateRepository((root / RESULTS_PATH) / "ioh_traces"),
        code_reader=FilesystemCodeReader(root),
        planned_target_conditions=baseline_targets,
    )
    audit = BenchmarkAuditService(
        workload, synthesis_reader, qwen_plan, BenchmarkLogger(verbose=False)
    ).get_audit_data(
        planned_models=planned_models,
        classical_baselines=baseline_ids,
        noise_robustness_enabled=False,
    )
    if not audit.is_success:
        raise RuntimeError(audit.error.message)
    audit = audit.value

    recorded_condition = tuple_(
        ExperimentORM.problem_id, ExperimentORM.dim, ExperimentORM.noise_std,
        ExperimentORM.mode, ExperimentORM.prompt_strategy,
    )
    filters = [
        and_(
            ExperimentORM.llm_name.in_(model_ids),
            recorded_condition.in_([
                (c.problem_id, c.dim, c.noise_std, c.mode, c.strategy) for c in plan
            ]),
        )
        for model_ids, plan in ((qwen_models, qwen_plan), ((luna_model,), luna_plan))
    ]
    ids_query = select(ExperimentORM.id).where(or_(*filters)).order_by(ExperimentORM.id)
    with engine.connect() as connection:
        history_ids = list(connection.execute(ids_query).scalars())
    synthesis_history = SQLiteSynthesisRepository(open_session).load_by_ids(history_ids)
    analysis = analyze_generations(synthesis_history)
finally:
    engine.dispose()

TypeError: BenchmarkAuditService.get_audit_data() got an unexpected keyword argument 'planned_models'

## Coverage by model

A **planned condition** fixes the problem, dimension, noise level, mode and strategy. Champion coverage counts conditions with a usable synthesized candidate. Full history means the recorded synthesis experiment contains all its candidate generations. Benchmark coverage uses the saved provenance for the planned native trials.

In [ ]:
counts = audit.coverage_by_model
model_counts = counts[counts["model"].isin(planned_models)]
model_labels = {"cmaes": "CMA-ES", "de": "DE"}
benchmark_labels = counts["model"].replace(model_labels)

overview = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.15,
    subplot_titles=("Planned synthesis coverage", "Planned native benchmark coverage"),
)
for name, key, color in (
    ("Planned", "planned_cells", "#b7c5d4"),
    ("Champion", "champion_cells", "#315d89"),
    ("Full history", "full_history_cells", "#188269"),
):
    overview.add_bar(
        x=model_counts[key], y=model_counts["model"], orientation="h",
        name=name, marker_color=color,
        customdata=model_counts["planned_cells"],
        hovertemplate="%{y}<br>%{x:,} of %{customdata:,} planned<extra>%{fullData.name}</extra>",
        row=1, col=1,
    )
for name, key, color in (
    ("Completed", "benchmark_completed_cells", "#188269"),
    ("Pending", "benchmark_pending_cells", "#a6b6c5"),
    ("Stale", "benchmark_stale_cells", "#b35826"),
    ("Invalid", "benchmark_invalid_cells", "#9e3b46"),
    ("Missing code", "benchmark_missing_code_cells", "#6d5ca1"),
):
    overview.add_bar(
        x=counts[key], y=benchmark_labels, orientation="h",
        name=name, marker_color=color,
        customdata=counts["planned_cells"],
        hovertemplate="%{y}<br>%{x:,} of %{customdata:,} planned<extra>%{fullData.name}</extra>",
        row=2, col=1,
    )
overview.update_xaxes(title_text="Conditions", rangemode="tozero")
overview.update_yaxes(autorange="reversed", automargin=True)
overview.update_layout(
    barmode="group", font_family=FONT_FAMILY, template="plotly_white",
    height=660, legend=dict(orientation="h", y=-0.12),
)
display(overview)

## Gaps to investigate

The tables show planned conditions that still need attention. Filter by model or classical solver with `GAP_MODEL` in the setup cell.

In [ ]:
coverage = audit.synthesis_coverage
synthesis_gaps = coverage[~(coverage["has_champion"] & coverage["has_full_history"])]
benchmark_gaps = audit.benchmarks[audit.benchmarks["status"] != "COMPLETED"]

if GAP_MODEL is not None:
    synthesis_gaps = synthesis_gaps[synthesis_gaps["model"] == GAP_MODEL]
    benchmark_gaps = benchmark_gaps[benchmark_gaps["model"] == GAP_MODEL]

condition_columns = ["model", "problem_id", "dim", "noise_std", "mode", "strategy"]
print(f"Synthesis gaps: {len(synthesis_gaps):,}")
if not synthesis_gaps.empty:
    display(
        synthesis_gaps[condition_columns + ["status", "reason"]]
        .sort_values(condition_columns, ignore_index=True)
    )

print(f"Native benchmark gaps: {len(benchmark_gaps):,}")
if not benchmark_gaps.empty:
    display(
        benchmark_gaps[
            condition_columns + ["runs_found", "target_runs", "status", "reason"]
        ].sort_values(condition_columns, ignore_index=True)
    )

## When did 10-generation experiments find their best candidate?

An **experiment** is one LLaMEA run for a fixed problem and configuration. A **generation** is one candidate attempt in that run. Each bar shows the percentage of included experiments whose lowest recorded error first occurred at that generation. Ties count at the earliest generation.

These plots use saved runs for the planned model and condition combinations above. A completed run with a full recorded history can be analyzed without a checkpoint.

In [ ]:
def show_generation_chart(analysis: GenerationAnalysis) -> None:
    ten = analysis.distribution[
        (analysis.distribution["generations"] == 10) & analysis.distribution["model"].isna()
    ]
    ten_runs = analysis.runs[analysis.runs["generations"] == 10]
    ten_exclusions = analysis.exclusions[analysis.exclusions["generations"] == 10]
    valid = int(ten_runs["best_generation"].count())
    no_score = len(ten_runs) - valid
    incomplete = int(ten_exclusions["reason"].eq("not_completed").sum())
    wrong_count = int(ten_exclusions["reason"].eq("iteration_count_mismatch").sum())
    coverage_note = (
        f"Included experiments: {valid:,}. Excluded from the 10-generation chart: "
        f"{incomplete:,} unfinished or failed, "
        f"{wrong_count:,} completed with missing or extra generation records, "
        f"and {no_score:,} completed with no valid score."
    )

    if not valid:
        print("No eligible experiments with a valid score to plot.")
        print(coverage_note)
        return

    generation_figure = go.Figure(
        go.Bar(
            x=ten["generation"],
            y=ten["best_fraction"],
            customdata=ten["best_count"],
            text=ten["best_fraction"],
            texttemplate="%{text:.1%}",
            textposition="outside",
            marker_color=["#315d89"] * 9 + ["#b94e21"],
            hovertemplate=(
                "Generation %{x}<br>%{customdata:,} of "
                f"{valid:,} experiments (%{{y:.1%}})<extra></extra>"
            ),
        )
    )
    generation_figure.update_xaxes(title="LLaMEA generation", dtick=1, range=[0.5, 10.5])
    generation_figure.update_yaxes(
        title="% of experiments",
        tickformat=".0%",
        range=[0, max(0.05, ten["best_fraction"].max() * 1.20)],
    )
    generation_figure.update_layout(
        title="Generation where the best candidate first appeared",
        height=410,
        **FIGURE_STYLE,
    )
    display(generation_figure)
    print(coverage_note)


show_generation_chart(analysis)

## Did generations 11–20 improve the result?

Each dot represents one completed 20-generation experiment with a valid score at both cutoffs. On the left, dots below the diagonal improved. On the right, a larger positive difference means a larger drop in error.

In [ ]:
def show_paired_comparison(analysis: GenerationAnalysis) -> None:
    pairs = analysis.pairs.dropna(subset=["error_at_10", "error_at_20"])

    if pairs.empty:
        print("No completed 20-generation experiments have valid scores at both cutoffs.")
        return

    paired_figure = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Best error at each cutoff", "Absolute improvement"),
    )
    for model, group in pairs.groupby("model", sort=True):
        trace_style = dict(
            mode="markers",
            marker_color=get_solver_color(model),
            name=model,
            legendgroup=model,
        )
        paired_figure.add_scatter(
            x=group["error_at_10"],
            y=group["error_at_20"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>10: %{x:.4g}<br>20: %{y:.4g}<extra></extra>",
            row=1,
            col=1,
            **trace_style,
        )
        paired_figure.add_scatter(
            x=group["experiment_id"],
            y=group["absolute_improvement"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>Improvement: %{y:.4g}<extra></extra>",
            showlegend=False,
            row=1,
            col=2,
            **trace_style,
        )
    error_limit = pairs["error_at_10"].max()
    paired_figure.add_shape(
        type="line", x0=0, y0=0, x1=error_limit, y1=error_limit,
        line=dict(color="#64748B", dash="dot"), row=1, col=1,
    )
    paired_figure.update_xaxes(title_text="Best error at generation 10", row=1, col=1)
    paired_figure.update_yaxes(title_text="Best error at generation 20", row=1, col=1)
    paired_figure.update_xaxes(title_text="Experiment ID", row=1, col=2)
    paired_figure.update_yaxes(title_text="Error at 10 minus error at 20", row=1, col=2)
    pooled = analysis.paired_summary.loc[analysis.paired_summary["model"].isna()].iloc[0]
    print(
        f"{pooled['improved']:,} of {pooled['comparable']:,} comparable experiments improved "
        f"({pooled['improved_fraction']:.1%}). Median error reduction across those "
        f"comparable experiments: {pooled['median_absolute_improvement']:.3g}."
    )
    paired_figure.update_layout(
        title="Error reduction from generations 11–20",
        height=490,
        **FIGURE_STYLE,
    )
    display(paired_figure)


show_paired_comparison(analysis)